##Incremental Data Loading using Autoloader

In [0]:
%sql
CREATE SCHEMA netflix_catalog.net_schema;

In [0]:
checkpoint_location = "abfss://silver@pravdatalake.dfs.core.windows.net/checkpoint/netflix"
schema_location = "abfss://silver@pravdatalake.dfs.core.windows.net/schema/netflix"

In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("cloudFiles.schemaLocation", schema_location)
    .load("abfss://raw@pravdatalake.dfs.core.windows.net")
)

In [0]:
display(df, checkpointLocation=f"{checkpoint_location}/display")

In [0]:
query = (
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_location)
    .outputMode("append")
    .trigger(availableNow=True)
    .start(
        "abfss://bronze@pravdatalake.dfs.core.windows.net/netflix_titles"
    )
)

query.awaitTermination()